# NEMOSIS Static Table Debugging Notebook

This notebook demonstrates how to use NEMOSIS directly to fetch static tables and helps debug issues with AEMO server responses.

## Problem
The cache file "NEW Registration and Exemption List.xls" contains HTML error content instead of Excel data: "Sorry, your request has failed. Please return to the home page and try again later."

## Debugging Steps
1. Test NEMOSIS static_table() function directly
2. Inspect cached files
3. Try alternative methods if AEMO server is having issues

In [ ]:
# Repository bootstrap for relocated notebooks
from pathlib import Path
import os
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
for _extra_path in (REPO_ROOT, REPO_ROOT / "src"):
    _extra_str = str(_extra_path)
    if _extra_str not in sys.path:
        sys.path.insert(0, _extra_str)

DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"
EVAL_OUTPUT_DIR = REPO_ROOT / "eval_output"
CONFIGS_DIR = REPO_ROOT / "configs"


In [ ]:
import os
import pandas as pd
from pathlib import Path

# Check if NEMOSIS is installed
try:
    import nemosis
    print(f"NEMOSIS version: {nemosis.__version__ if hasattr(nemosis, '__version__') else 'unknown'}")
    print(f"NEMOSIS location: {nemosis.__file__}")
except ImportError as e:
    print(f"ERROR: NEMOSIS not installed: {e}")
    print("Install with: pip install nemosis")

## 1. Check NEMOSIS Cache Directory

NEMOSIS stores cached files in a default location. Let's find and inspect it.

In [ ]:
# NEMOSIS default cache directory (usually in user's home)
# On Linux/Mac: ~/nemosis_cache or ~/.nemosis_cache
# On Windows: C:\Users\<username>\nemosis_cache

possible_cache_dirs = [
    Path.home() / "nemosis_cache",
    Path.home() / ".nemosis_cache",
    Path("nemosis_cache"),
    Path("data/aemo")  # Our custom cache dir
]

print("Searching for NEMOSIS cache directories:")
for cache_dir in possible_cache_dirs:
    if cache_dir.exists():
        print(f"\n✓ Found: {cache_dir}")
        print(f"  Contents:")
        for item in sorted(cache_dir.iterdir())[:20]:  # Show first 20 items
            size = item.stat().st_size if item.is_file() else 0
            print(f"    {item.name} ({size:,} bytes)")
    else:
        print(f"✗ Not found: {cache_dir}")

## 2. Inspect the Problematic Cache File

Let's look at the content of "NEW Registration and Exemption List.xls" to confirm it contains HTML error.

In [ ]:
# Try to find the problematic file
problem_filename = "NEW Registration and Exemption List.xls"

for cache_dir in possible_cache_dirs:
    if cache_dir.exists():
        problem_file = cache_dir / problem_filename
        if problem_file.exists():
            print(f"Found problematic file: {problem_file}")
            print(f"File size: {problem_file.stat().st_size:,} bytes")
            
            # Read first 2KB to see if it's HTML
            with open(problem_file, 'rb') as f:
                content = f.read(2048)
                
            print("\nFirst 2KB of file (as text):")
            try:
                print(content.decode('utf-8', errors='replace')[:1000])
            except:
                print(content[:1000])
            
            # Check if it's HTML
            content_lower = content.lower()
            is_html = b'<html' in content_lower or b'<!doctype' in content_lower
            has_error = b'sorry' in content_lower or b'failed' in content_lower
            
            print(f"\nIs HTML: {is_html}")
            print(f"Contains error message: {has_error}")
            
            if is_html and has_error:
                print("\n⚠️ CONFIRMED: File contains HTML error page, not Excel data!")
                print("   This is why pandas fails to read it.")
                print("\n   To fix: Delete this file and let NEMOSIS re-download it:")
                print(f"   rm \"{problem_file}\"")
            break

## 3. Test NEMOSIS static_table() Function Directly

Let's try calling NEMOSIS directly to fetch the static table.

In [ ]:
from nemosis import static_table

print("Attempting to fetch 'Generators and Scheduled Loads' table from NEMOSIS...")
print("This may take 1-2 minutes if downloading for the first time.\n")

try:
    # Method 1: Try with default cache directory
    generators = static_table(
        table_name='Generators and Scheduled Loads',
        raw_cache=Path.home() / 'nemosis_cache'
    )
    
    print("✓ SUCCESS! Table fetched.")
    print(f"\nTable shape: {generators.shape}")
    print(f"\nColumns ({len(generators.columns)}):")
    print(generators.columns.tolist())
    print(f"\nFirst few rows:")
    print(generators.head())
    
    # Check for fuel type columns
    fuel_cols = [col for col in generators.columns if 'fuel' in col.lower() or 'source' in col.lower()]
    print(f"\nFuel-related columns: {fuel_cols}")
    
except Exception as e:
    print(f"✗ ERROR fetching table: {type(e).__name__}: {e}")
    print("\nPossible reasons:")
    print("1. AEMO server is temporarily unavailable (returns HTML error page)")
    print("2. Network connectivity issues")
    print("3. Cached file is corrupted (contains HTML error)")
    print("\nTry:")
    print("- Wait a few minutes and retry")
    print("- Delete cache directory and retry")
    print(f"  rm -rf ~/nemosis_cache")

## 4. Test with Force Refresh

Let's try forcing a fresh download by deleting the cached file first.

In [ ]:
import shutil

# Delete the problematic cache file if it exists
cache_dir = Path.home() / 'nemosis_cache'
problem_file = cache_dir / problem_filename

if problem_file.exists():
    print(f"Deleting corrupted cache file: {problem_file}")
    problem_file.unlink()
    print("✓ Deleted")
else:
    print(f"File not found: {problem_file}")

print("\nAttempting fresh download...")
try:
    generators = static_table(
        table_name='Generators and Scheduled Loads',
        raw_cache=cache_dir
    )
    print("✓ SUCCESS with fresh download!")
    print(f"Table shape: {generators.shape}")
    
    # Verify the new file is valid Excel (not HTML)
    if problem_file.exists():
        with open(problem_file, 'rb') as f:
            content = f.read(1024)
        is_html = b'<html' in content.lower() or b'<!doctype' in content.lower()
        if is_html:
            print("\n⚠️ WARNING: Newly downloaded file is STILL HTML error page!")
            print("   AEMO server is likely having issues right now.")
            print("   Try again in 10-30 minutes.")
        else:
            print("\n✓ Newly downloaded file appears to be valid Excel data.")
            
except Exception as e:
    print(f"✗ ERROR even with fresh download: {e}")
    print("\nThis suggests AEMO's server is currently returning error pages.")
    print("Please try again later (typically within 30 minutes to a few hours).")

## 5. Alternative: List Available Static Tables

Let's see what static tables NEMOSIS knows about.

In [ ]:
# NEMOSIS doesn't provide a list function, but we can try common table names
common_tables = [
    'Generators and Scheduled Loads',
    'DUDETAILSUMMARY',
    'DUDETAIL',
    'Transmission Lines and Interconnectors',
    'GENUNITS',
]

print("Testing common NEMOSIS static table names:\n")
for table_name in common_tables:
    try:
        df = static_table(table_name=table_name, raw_cache=cache_dir)
        print(f"✓ {table_name}: {df.shape[0]} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"✗ {table_name}: {type(e).__name__}")

## 6. Test DISPATCH_UNIT_SCADA (Dynamic Data)

Let's verify that dynamic data fetching works (this is different from static tables).

In [ ]:
from nemosis import dynamic_data_compiler

start_time = '2024/01/01 00:00:00'
end_time = '2024/01/01 01:00:00'

print(f"Testing dynamic data fetch: DISPATCH_UNIT_SCADA")
print(f"Date range: {start_time} to {end_time}\n")

try:
    scada_data = dynamic_data_compiler(
        start_time=start_time,
        end_time=end_time,
        table_name='DISPATCH_UNIT_SCADA',
        raw_data_cache=cache_dir
    )
    
    print("✓ SUCCESS! DISPATCH_UNIT_SCADA data fetched.")
    print(f"\nData shape: {scada_data.shape}")
    print(f"\nColumns: {scada_data.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(scada_data.head())
    
    # Count unique DUIDs
    if 'DUID' in scada_data.columns:
        print(f"\nUnique DUIDs in this sample: {scada_data['DUID'].nunique()}")
        print(f"Example DUIDs: {scada_data['DUID'].unique()[:10].tolist()}")
    
except Exception as e:
    print(f"✗ ERROR: {e}")
    print("\nDynamic data fetching failed. This could indicate:")
    print("1. AEMO server issues")
    print("2. Invalid date range")
    print("3. Network connectivity problems")

## Summary and Recommendations

Run all cells above to diagnose the issue. Based on the results:

### If static table fetch FAILS:
1. **AEMO server is returning HTML error pages** (not Excel files)
2. **Solution**: Wait 10-60 minutes and retry - this is a temporary server issue
3. **Workaround**: Use dynamic data only (skip generation by fuel type for now)

### If you need to proceed without static tables:
You can still fetch dispatch prices, FCAS prices, and unit dispatch data. Only `fetch_aemo_generation_by_fuel()` requires the static table.

### To manually clean cache:
```bash
# Delete entire NEMOSIS cache
rm -rf ~/nemosis_cache

# Or just the problematic file
rm ~/nemosis_cache/"NEW Registration and Exemption List.xls"
```